# Technical Q&A Assistant

A full-featured prototype featuring:
- **Gradio UI** with streaming output
- **LiteLLM** for multi-model switching (GPT-4o, DeepSeek-R1, Llama 3.2, Gemma 3, etc.)
- **Tool use** – live web search via DuckDuckGo
- **🎙️ Audio input** (speech-to-text via Whisper)
- **🔊 Audio output** (text-to-speech via OpenAI TTS)


---
## Architecture Overview

```
User (browser)
    │
    ▼
Gradio UI  ──── text/audio input ────►  transcribe_audio()  (Whisper)
    │                                          │
    │◄──── streaming tokens ─────  chat_stream()  (LiteLLM)
    │                                   │
    │              ┌────────────────────┘
    │              │   tool_calls detected?
    │              ▼
    │         run_web_search()  (DuckDuckGo)
    │              │
    │              └──► inject tool results → second LLM call
    │
    ▼
text_to_speech()  (OpenAI TTS)  →  audio player
```

| Feature | Implementation |
|---|---|
| Multi-model | LiteLLM unified interface |
| Open Source models | Ollama (`deepseek-r1:1.5b`, `llama3.2`, `gemma3:270m`) |
| Frontier models | OpenAI GPT-4o / GPT-4o Mini |
| Streaming | `litellm.completion(..., stream=True)` |
| Tool use | OpenAI function-calling spec + DuckDuckGo (cloud only) |
| Speech-to-text | OpenAI Whisper v1 |
| Text-to-speech | OpenAI TTS (`tts-1`, 6 voices) |
| UI | Gradio Blocks with Soft theme |


## API Key Setup

In [ ]:
import json, tempfile, pathlib
import litellm
from openai import OpenAI
from duckduckgo_search import DDGS
import gradio as gr
import os
from dotenv import load_dotenv


In [ ]:
load_dotenv(override=True)

open_ai_key = os.getenv("OPENAI_API_KEY")

## Core Logic

In [ ]:
litellm.drop_params = True   # silently ignore unsupported params per model

# Models exposed in the UI 
OLLAMA_BASE_URL = "http://localhost:11434"

MODELS = {
    # Frontier models
    "GPT-4o (OpenAI)":           "gpt-4o",
    "GPT-4o Mini (OpenAI)":      "gpt-4o-mini",
    # Open-source via Ollama 
    "DeepSeek-R1 1.5B (Ollama)": "ollama/deepseek-r1:1.5b",
    "Llama 3.2 (Ollama)":        "ollama/llama3.2:latest",
    "Gemma 3 270M (Ollama)":     "ollama/gemma3:270m",
}


# Models that support OpenAI-style function/tool calling
TOOL_CAPABLE_MODELS = {"gpt-4o", "gpt-4o-mini"}


In [ ]:
# Expert system prompt
SYSTEM_PROMPT = """
You are **TechSage**, a world-class technical expert and AI engineer with deep knowledge of:
• Machine learning, LLMs, prompt engineering, and AI systems
• Software engineering (Python, JavaScript, Go, Rust, and more)
• Cloud infrastructure, DevOps, and distributed systems
• Mathematics, algorithms, and data structures

Guidelines:
1. Give clear, accurate, expert-level answers with working code examples when helpful.
2. If you use the `web_search` tool, always synthesise the results into your answer — do not just dump links.
3. Structure complex answers with headers and bullet points.
4. Be concise but complete; never sacrifice correctness for brevity.
5. If a question is ambiguous, briefly state your interpretation before answering.
6. If you are asked a question that is not related to technology, say so.
6. Always give concise answers limited to 250 words.
""".strip()


In [ ]:

# Tool definition (DuckDuckGo web search) 
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "web_search",
            "description": "Search the web for up-to-date technical information. Use this when you need current docs, changelogs, error messages, or recent events.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "The search query"},
                    "max_results": {"type": "integer", "default": 5, "description": "Number of results to return"}
                },
                "required": ["query"]
            }
        }
    }
]



In [ ]:
def run_web_search(query: str, max_results: int = 5) -> str:
    """Execute a DuckDuckGo search and return formatted results."""
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=max_results))
        if not results:
            return "No results found."
        lines = []
        for i, r in enumerate(results, 1):
            lines.append(f"{i}. **{r.get('title','')}**\n   {r.get('href','')}\n   {r.get('body','')}")
        return "\n\n".join(lines)
    except Exception as e:
        return f"Search error: {e}"

## Streaming Chat with Tool Use

In [ ]:
def chat_stream(user_message: str, history: list, model_label: str, use_tools: bool, temperature: float):
    """
    Yields (history, status_text) tuples as tokens arrive.
    Handles tool calls transparently (DuckDuckGo web search).
    """
    model = MODELS[model_label]
    is_ollama = model.startswith("ollama/")
    supports_tools = model in TOOL_CAPABLE_MODELS
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    # Reconstruct conversation from Gradio history
    for h in history:
        messages.append({"role": "user",      "content": h[0]})
        messages.append({"role": "assistant", "content": h[1]})

    messages.append({"role": "user", "content": user_message})

    # Append the new empty assistant turn so Gradio shows typing indicator
    history = history + [[user_message, ""]]

    kwargs = dict(model=model, messages=messages, temperature=temperature, stream=True)
    if is_ollama:
        kwargs["api_base"] = OLLAMA_BASE_URL
    if use_tools and supports_tools:
        kwargs["tools"] = TOOLS
    elif use_tools and not supports_tools:
        # Inject web-search hint into system prompt for non-tool models
        messages[0]["content"] += (
            "\n\nNote: you do not have live web access on this model. "
            "If asked to search the web, say so honestly and answer from your training data."
        )

    status = f"🤖 Using **{model_label}**"
    yield history, status

    # First streaming pass 
    collected_chunks = []
    tool_call_accumulator = {}   # id -> {name, arguments str}
    finish_reason = None

    try:
        response = litellm.completion(**kwargs)
        for chunk in response:
            delta = chunk.choices[0].delta
            finish_reason = chunk.choices[0].finish_reason

            # Accumulate regular text
            if delta.content:
                history[-1][1] += delta.content
                yield history, status

            # Accumulate tool call fragments
            if hasattr(delta, "tool_calls") and delta.tool_calls:
                for tc in delta.tool_calls:
                    tid = tc.index
                    if tid not in tool_call_accumulator:
                        tool_call_accumulator[tid] = {"id": tc.id or str(tid), "name": "", "arguments": ""}
                    if tc.function.name:
                        tool_call_accumulator[tid]["name"] += tc.function.name
                    if tc.function.arguments:
                        tool_call_accumulator[tid]["arguments"] += tc.function.arguments

    except Exception as e:
        history[-1][1] = f"❌ Error: {e}"
        yield history, "❌ Error"
        return

    # Handle tool calls 
    if tool_call_accumulator:
        tool_results_msgs = []

        for tid, tc in tool_call_accumulator.items():
            if tc["name"] == "web_search":
                try:
                    args = json.loads(tc["arguments"])
                except Exception:
                    args = {"query": tc["arguments"]}

                query = args.get("query", "")
                max_r = args.get("max_results", 5)
                status = f"🔍 Searching: *{query}*"
                history[-1][1] = f"*🔍 Searching the web for: `{query}`…*"
                yield history, status

                search_result = run_web_search(query, max_r)
                tool_results_msgs.append({
                    "role": "tool",
                    "tool_call_id": tc["id"],
                    "name": "web_search",
                    "content": search_result
                })

        # Second call with tool results (streaming)
        messages.append({
            "role": "assistant",
            "content": None,
            "tool_calls": [
                {"id": tc["id"], "type": "function",
                 "function": {"name": tc["name"], "arguments": tc["arguments"]}}
                for tc in tool_call_accumulator.values()
            ]
        })
        messages.extend(tool_results_msgs)

        history[-1][1] = ""  # reset to stream the real answer
        status = f"🤖 Synthesising answer with **{model_label}**"

        try:
            resp2_kwargs = dict(model=model, messages=messages,
                               temperature=temperature, stream=True)
            if is_ollama:
                resp2_kwargs["api_base"] = OLLAMA_BASE_URL
            response2 = litellm.completion(**resp2_kwargs)
            for chunk in response2:
                delta = chunk.choices[0].delta
                if delta.content:
                    history[-1][1] += delta.content
                    yield history, status
        except Exception as e:
            history[-1][1] += f"\n\n❌ Error in second pass: {e}"

    yield history, f"✅ Done — {model_label}"

## Audio Helpers (Whisper STT + OpenAI TTS)

In [ ]:
openai_client = OpenAI()  # uses OPENAI_API_KEY from env

def transcribe_audio(audio_path: str | None) -> str:
    """Convert recorded audio to text using OpenAI Whisper."""
    if not audio_path:
        return ""
    try:
        with open(audio_path, "rb") as f:
            transcript = openai_client.audio.transcriptions.create(
                model="whisper-1", file=f
            )
        return transcript.text
    except Exception as e:
        return f"[Transcription error: {e}]"



In [ ]:
def text_to_speech(text: str, voice: str = "nova") -> str | None:
    """Convert text to an MP3 file using OpenAI TTS. Returns the file path."""
    if not text or len(text.strip()) < 3:
        return None
    # Truncate very long responses to keep TTS snappy
    snippet = text[:1200] + ("…" if len(text) > 1200 else "")
    try:
        tmp = tempfile.NamedTemporaryFile(suffix=".mp3", delete=False)
        with openai_client.audio.speech.with_streaming_response.create(
            model="tts-1", voice=voice, input=snippet
        ) as resp:
            resp.stream_to_file(tmp.name)
        return tmp.name
    except Exception as e:
        print(f"TTS error: {e}")
        return None

## Gradio UI

In [ ]:
VOICES = ["alloy", "echo", "fable", "onyx", "nova", "shimmer"]

EXAMPLE_QUESTIONS = [
    "What is the difference between RAG and fine-tuning? When should I use each?",
    "Explain LiteLLM and show me how to call three different providers with the same code.",
    "What are the pros and cons of async vs threaded web servers in Python?",
    "Search the web: what's new in the latest version of Gradio?",
    "Walk me through transformer attention with a worked numerical example.",
]

def handle_audio_input(audio_path, current_text):
    """Transcribe audio and append to text box."""
    transcribed = transcribe_audio(audio_path)
    if transcribed and not transcribed.startswith("[Transcription error"):
        return (current_text + " " + transcribed).strip()
    return current_text


In [ ]:
def user_submit(user_msg, history, model_label, use_tools, temperature, tts_enabled, voice):
    """Wrapper: stream chat then optionally synthesise audio."""
    if not user_msg.strip():
        yield history, "", "Please enter a question.", None
        return

    final_history = history
    for final_history, status in chat_stream(user_msg, history, model_label, use_tools, temperature):
        yield final_history, "", status, None  # clear input, no audio yet

    # Generate TTS for the last assistant turn
    audio_path = None
    if tts_enabled and final_history:
        last_answer = final_history[-1][1]
        audio_path = text_to_speech(last_answer, voice)

    yield final_history, "", status, audio_path

In [ ]:
CUSTOM_CSS = """
@import url('https://fonts.googleapis.com/css2?family=Space+Grotesk:wght@300;400;500;600;700&family=JetBrains+Mono:wght@400;500&display=swap');

/* ═══════════════════════════════════════════════
   PALETTE
═══════════════════════════════════════════════ */
:root {
    --ts-bg:       #0d0f1a;
    --ts-surface:  #13162b;
    --ts-card:     #191c33;
    --ts-border:   #2e3158;
    --ts-accent:   #7c6af7;
    --ts-accent2:  #a78bfa;
    --ts-teal:     #34d399;
    --ts-text:     #dde1f5;
    --ts-subtext:  #9ba3cc;
    --ts-muted:    #5c6290;
    --ts-glow:     rgba(124,106,247,0.20);
    --ts-radius:   12px;
    --ts-radius-s: 8px;
}

/* ═══════════════════════════════════════════════
   GLOBAL CONTAINER
═══════════════════════════════════════════════ */
body,
.gradio-container,
.gradio-container > .main,
.gradio-container > .main > .wrap,
footer {
    background: var(--ts-bg) !important;
    color: var(--ts-text) !important;
    font-family: 'Space Grotesk', sans-serif !important;
}
/* kill Gradio's white panel flash */
.gr-panel, .gr-box, .gap, .form,
div.svelte-1gfkn6j, div.svelte-10ogue4 {
    background: transparent !important;
}

/* ═══════════════════════════════════════════════
   HEADER
═══════════════════════════════════════════════ */
.ts-header {
    background: linear-gradient(135deg, #141627 0%, #1a1335 50%, #0e1a2e 100%);
    border: 1px solid var(--ts-border);
    border-radius: 16px;
    padding: 26px 34px;
    margin-bottom: 10px;
    position: relative;
    overflow: hidden;
}
.ts-header::before {
    content: '';
    position: absolute; inset: 0;
    background:
        radial-gradient(ellipse at 15% 60%, rgba(124,106,247,0.13) 0%, transparent 55%),
        radial-gradient(ellipse at 85% 20%, rgba(52,211,153,0.08) 0%, transparent 50%);
    pointer-events: none;
}
.ts-header h1 {
    font-size: 1.85rem !important;
    font-weight: 700 !important;
    background: linear-gradient(90deg, #a78bfa 0%, #7c6af7 50%, #34d399 100%) !important;
    -webkit-background-clip: text !important;
    -webkit-text-fill-color: transparent !important;
    background-clip: text !important;
    margin: 0 0 8px !important;
    letter-spacing: -0.02em;
}
.ts-header p {
    color: var(--ts-subtext) !important;
    font-size: 0.85rem !important;
    font-family: 'JetBrains Mono', monospace !important;
    margin: 0 !important;
}
.badge {
    display: inline-block;
    background: rgba(124,106,247,0.15);
    border: 1px solid rgba(124,106,247,0.32);
    color: #c4b5fd;
    border-radius: 20px;
    padding: 2px 9px;
    font-size: 0.72rem;
    font-weight: 600;
    margin: 0 2px;
    vertical-align: middle;
}
.badge.teal {
    background: rgba(52,211,153,0.12);
    border-color: rgba(52,211,153,0.32);
    color: #6ee7b7;
}

/* ═══════════════════════════════════════════════
   ACCORDION
═══════════════════════════════════════════════ */
.gr-accordion, details.gr-accordion {
    background: var(--ts-card) !important;
    border: 1px solid var(--ts-border) !important;
    border-radius: var(--ts-radius) !important;
    margin-bottom: 8px !important;
    overflow: hidden;
}
.gr-accordion > .label-wrap,
details.gr-accordion > summary {
    background: transparent !important;
    color: var(--ts-text) !important;
    font-weight: 600 !important;
    font-size: 0.87rem !important;
    padding: 13px 16px !important;
    cursor: pointer !important;
    user-select: none;
    border-bottom: 1px solid transparent;
    transition: background 0.15s;
}
details.gr-accordion[open] > summary {
    border-bottom-color: var(--ts-border);
}
.gr-accordion > .label-wrap:hover,
details.gr-accordion > summary:hover {
    background: rgba(124,106,247,0.08) !important;
}
.gr-accordion .inner,
details.gr-accordion > div {
    padding: 12px 14px 14px !important;
    background: var(--ts-card) !important;
}
.gr-accordion svg, details.gr-accordion summary svg {
    stroke: var(--ts-subtext) !important;
    fill: none !important;
}

/* ═══════════════════════════════════════════════
   LABELS
═══════════════════════════════════════════════ */
label,
.gr-form label,
span.svelte-1ed2p3z,
.wrap > label,
.label-wrap span,
.gr-block label,
.block label span,
p.svelte-1ed2p3z,
.gr-compact label {
    color: var(--ts-subtext) !important;
    font-size: 0.82rem !important;
    font-weight: 500 !important;
    font-family: 'Space Grotesk', sans-serif !important;
}

/* ═══════════════════════════════════════════════
   TEXT INPUTS
═══════════════════════════════════════════════ */
textarea,
input[type="text"],
input[type="search"],
input[type="number"] {
    background: #1e2240 !important;
    border: 1px solid var(--ts-border) !important;
    border-radius: var(--ts-radius-s) !important;
    color: #dde1f5 !important;
    -webkit-text-fill-color: #dde1f5 !important;
    font-family: 'Space Grotesk', sans-serif !important;
    font-size: 0.92rem !important;
    caret-color: var(--ts-accent2);
    transition: border-color 0.2s, box-shadow 0.2s !important;
}
textarea::placeholder, input::placeholder {
    color: #5c6290 !important;
    -webkit-text-fill-color: #5c6290 !important;
    opacity: 1 !important;
}
textarea:focus,
input[type="text"]:focus,
input[type="number"]:focus {
    border-color: var(--ts-accent) !important;
    box-shadow: 0 0 0 3px var(--ts-glow) !important;
    outline: none !important;
}
/* wrapper blocks must not impose their own bg/color */
.gr-text-input, .block.svelte-90oupt,
.gr-padded, .wrap.svelte-1ed2p3z,
.gr-textbox, div[data-testid="textbox"],
div[data-testid="textbox"] > label,
div[data-testid="textbox"] > label > textarea {
    background: transparent !important;
    color: #dde1f5 !important;
    -webkit-text-fill-color: #dde1f5 !important;
}

/* ═══════════════════════════════════════════════
   DROPDOWN / SELECT
═══════════════════════════════════════════════ */
.gr-dropdown, .wrap.svelte-qjvbzn, ul.options {
    background: var(--ts-surface) !important;
    border-color: var(--ts-border) !important;
    color: var(--ts-text) !important;
}
.gr-dropdown input, .wrap.svelte-qjvbzn input {
    background: transparent !important;
    color: var(--ts-text) !important;
}
ul.options li, .option-string {
    background: var(--ts-card) !important;
    color: var(--ts-text) !important;
}
ul.options li:hover, ul.options li.active {
    background: rgba(124,106,247,0.18) !important;
    color: #fff !important;
}

/* ═══════════════════════════════════════════════
   SLIDER
═══════════════════════════════════════════════ */
input[type="range"] { accent-color: var(--ts-accent) !important; }
.gr-slider .range-value, .gr-slider output, .gr-slider span {
    color: var(--ts-subtext) !important;
    background: transparent !important;
}
.gr-form .info, .block .info, span.info.svelte-1ed2p3z {
    color: var(--ts-muted) !important;
    font-size: 0.77rem !important;
}

/* ═══════════════════════════════════════════════
   CHECKBOX
═══════════════════════════════════════════════ */
input[type="checkbox"] {
    accent-color: var(--ts-accent) !important;
    width: 15px; height: 15px;
}
.gr-checkbox-group label, .gr-checkbox label, .checkbox-group label {
    color: var(--ts-text) !important;
}

/* ═══════════════════════════════════════════════
   BUTTONS
═══════════════════════════════════════════════ */
button.primary, .gr-button-primary, button[class*="primary"] {
    background: linear-gradient(135deg, #7c6af7 0%, #5b4fd4 100%) !important;
    border: none !important;
    color: #ffffff !important;
    font-weight: 600 !important;
    font-family: 'Space Grotesk', sans-serif !important;
    font-size: 0.9rem !important;
    border-radius: var(--ts-radius-s) !important;
    box-shadow: 0 4px 14px rgba(124,106,247,0.4) !important;
    transition: transform 0.15s, box-shadow 0.15s !important;
    letter-spacing: 0.02em;
}
button.primary:hover  { transform: translateY(-1px) !important; box-shadow: 0 6px 22px rgba(124,106,247,0.55) !important; }
button.primary:active { transform: translateY(0) !important; }

button.secondary, .gr-button-secondary, button[class*="secondary"] {
    background: rgba(124,106,247,0.10) !important;
    border: 1px solid rgba(124,106,247,0.35) !important;
    color: var(--ts-accent2) !important;
    font-weight: 500 !important;
    font-family: 'Space Grotesk', sans-serif !important;
    font-size: 0.88rem !important;
    border-radius: var(--ts-radius-s) !important;
    transition: background 0.15s, border-color 0.15s !important;
}
button.secondary:hover {
    background: rgba(124,106,247,0.20) !important;
    border-color: var(--ts-accent) !important;
    color: #fff !important;
}

/* Example question buttons */
.example-btn button, .example-btn button.secondary {
    text-align: left !important;
    justify-content: flex-start !important;
    font-size: 0.81rem !important;
    padding: 8px 11px !important;
    width: 100% !important;
    background: rgba(52,211,153,0.07) !important;
    border: 1px solid rgba(52,211,153,0.22) !important;
    color: #86efcb !important;
    border-radius: 8px !important;
    margin-bottom: 5px !important;
}
.example-btn button:hover {
    background: rgba(52,211,153,0.15) !important;
    border-color: rgba(52,211,153,0.45) !important;
    color: #d1fae5 !important;
}

/* ═══════════════════════════════════════════════
   CHATBOT
═══════════════════════════════════════════════ */
.chatbot, div[data-testid="chatbot"], .chatbot > div {
    background: var(--ts-surface) !important;
    border: 1px solid var(--ts-border) !important;
    border-radius: var(--ts-radius) !important;
}
/* Every text node inside the chat viewport */
.chatbot *,
div[data-testid="chatbot"] * {
    color: #dde1f5 !important;
    -webkit-text-fill-color: #dde1f5 !important;
}
/* Bot bubble */
.message.bot,
div[data-testid="bot"],
div[data-testid="bot"] > div,
.bot > div,
.prose {
    background: #191c33 !important;
    border: 1px solid #2e3158 !important;
    color: #dde1f5 !important;
    -webkit-text-fill-color: #dde1f5 !important;
}
/* User bubble */
.message.user,
div[data-testid="user"],
div[data-testid="user"] > div,
.user > div {
    background: linear-gradient(135deg, rgba(124,106,247,0.26), rgba(167,139,250,0.14)) !important;
    border: 1px solid rgba(124,106,247,0.40) !important;
    color: #dde1f5 !important;
    -webkit-text-fill-color: #dde1f5 !important;
}
/* Markdown rendered content inside bubbles */
.message p, .message li, .message h1, .message h2,
.message h3, .message h4, .message span, .message a,
.prose p, .prose li, .prose h1, .prose h2, .prose h3,
.prose span, .prose a,
div[data-testid="bot"] p,
div[data-testid="bot"] li,
div[data-testid="bot"] span,
div[data-testid="user"] p,
div[data-testid="user"] span {
    color: #dde1f5 !important;
    -webkit-text-fill-color: #dde1f5 !important;
}
/* Code blocks inside chat */
.message code, .message pre,
.prose code, .prose pre,
div[data-testid="bot"] code,
div[data-testid="bot"] pre {
    background: rgba(13,15,26,0.80) !important;
    color: #c4b5fd !important;
    -webkit-text-fill-color: #c4b5fd !important;
    border-radius: 5px;
    font-family: 'JetBrains Mono', monospace !important;
}
/* Generic message wrapper */
.message {
    font-family: 'Space Grotesk', sans-serif !important;
    font-size: 0.92rem !important;
    line-height: 1.65 !important;
    color: #dde1f5 !important;
    -webkit-text-fill-color: #dde1f5 !important;
}

/* ═══════════════════════════════════════════════
   STATUS BAR
═══════════════════════════════════════════════ */
.status-bar,
.status-bar > div,
.status-bar p,
.status-bar em,
.status-bar * {
    background: var(--ts-surface) !important;
    color: #9ba3cc !important;
    -webkit-text-fill-color: #9ba3cc !important;
    font-family: 'JetBrains Mono', monospace !important;
    font-size: 0.77rem !important;
}
.status-bar {
    border: 1px solid var(--ts-border) !important;
    border-radius: 6px !important;
    padding: 5px 12px !important;
    margin-top: 5px !important;
}

/* ═══════════════════════════════════════════════
   AUDIO COMPONENT
═══════════════════════════════════════════════ */
.gr-audio,
div[data-testid="audio"],
div[data-testid="audio"] > div {
    background: var(--ts-surface) !important;
    border: 1px solid var(--ts-border) !important;
    border-radius: var(--ts-radius) !important;
    color: var(--ts-text) !important;
}
div[data-testid="audio"] label,
div[data-testid="audio"] span {
    color: var(--ts-subtext) !important;
    background: transparent !important;
}
div[data-testid="audio"] button {
    color: var(--ts-accent2) !important;
    background: rgba(124,106,247,0.12) !important;
    border: 1px solid rgba(124,106,247,0.3) !important;
    border-radius: 6px !important;
}
div[data-testid="audio"] button:hover {
    background: rgba(124,106,247,0.22) !important;
}
div[data-testid="audio"] svg {
    fill: var(--ts-accent2) !important;
    stroke: var(--ts-accent2) !important;
}

/* ═══════════════════════════════════════════════
   MISC
═══════════════════════════════════════════════ */
.wrap, .block, .form, .container { background: transparent !important; }
::-webkit-scrollbar { width: 5px; height: 5px; }
::-webkit-scrollbar-track { background: var(--ts-bg); }
::-webkit-scrollbar-thumb { background: var(--ts-border); border-radius: 3px; }
::-webkit-scrollbar-thumb:hover { background: var(--ts-accent); }
footer { display: none !important; }
"""


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#  UI Layout
# ─────────────────────────────────────────────────────────────────────────────
with gr.Blocks(
    theme=gr.themes.Base(
        primary_hue="violet",
        secondary_hue="indigo",
        neutral_hue="slate",
        font=gr.themes.GoogleFont("Space Grotesk"),
        font_mono=gr.themes.GoogleFont("JetBrains Mono"),
    ),
    css=CUSTOM_CSS,
    title="TechSage – Technical Q&A Assitant",
) as demo:

    # Header 
    gr.HTML("""
    <div class="ts-header">
        <h1>🤖 TechSage</h1>
        <p>
            Technical Q&amp;A Assistant&nbsp;&nbsp;
            <span class="badge">LiteLLM</span>
            <span class="badge">Streaming</span>
            <span class="badge teal">🔍 Web Search</span>
            <span class="badge teal">🔊 Voice I/O</span>
        </p>
    </div>
    """)

    with gr.Row(equal_height=False):

        # Left sidebar 
        with gr.Column(scale=1, min_width=270):

            with gr.Accordion("⚙️  Model & Settings", open=False):
                model_dd = gr.Dropdown(
                    label="Model",
                    choices=list(MODELS.keys()),
                    value="GPT-4o (OpenAI)",
                )
                temperature_sl = gr.Slider(
                    label="Temperature",
                    minimum=0.0, maximum=1.5, step=0.05, value=0.3,
                    info="Lower = focused  ·  Higher = creative",
                )
                use_tools_cb = gr.Checkbox(
                    label="🔍 Enable Web Search (cloud models only)",
                    value=True,
                )

            with gr.Accordion("🔊  Voice Settings", open=False):
                tts_cb   = gr.Checkbox(label="Read answers aloud (TTS)", value=False)
                voice_dd = gr.Dropdown(
                    label="TTS Voice", choices=VOICES, value="nova"
                )

            with gr.Accordion("💡  Example Questions", open=False):
                gr.HTML(
                    "<p style='color:#5c6290;font-size:0.78rem;"
                    "font-family:JetBrains Mono,monospace;margin:0 0 10px;'>"
                    "↓ Click to load into the chat box</p>"
                )
                example_btns = []
                for ex in EXAMPLE_QUESTIONS:
                    with gr.Row(elem_classes="example-btn"):
                        btn = gr.Button(
                            ex[:72] + ("…" if len(ex) > 72 else ""),
                            size="sm", variant="secondary"
                        )
                        example_btns.append((btn, ex))

        # Main chat area 
        with gr.Column(scale=3):

            chatbot = gr.Chatbot(
                label="TechSage",
                height=500,
                bubble_full_width=False,
                render_markdown=True,
                show_copy_button=True,
                avatar_images=(
                    None,
                    "https://api.dicebear.com/7.x/bottts/svg?seed=techsage",
                ),
            )

            with gr.Row():
                msg_box = gr.Textbox(
                    show_label=False,
                    placeholder="Ask anything technical…  or record audio below 👇",
                    lines=3,
                    scale=5,
                )
                with gr.Column(scale=1, min_width=120):
                    send_btn  = gr.Button("Send 🚀",  variant="primary")
                    clear_btn = gr.Button("Clear 🗑️", variant="secondary")

            with gr.Accordion("🎙️  Voice Input — record & transcribe", open=False):
                audio_in = gr.Audio(
                    sources=["microphone"], type="filepath",
                    label="Record your question",
                )
                transcribe_btn = gr.Button("Transcribe 🎤", variant="secondary")

            status_md = gr.Markdown("*Ready*", elem_classes="status-bar")

            audio_out = gr.Audio(
                label="🔊 Answer Audio", autoplay=True, visible=True
            )

    # Event wiring 
    submit_inputs  = [msg_box, chatbot, model_dd, use_tools_cb, temperature_sl, tts_cb, voice_dd]
    submit_outputs = [chatbot, msg_box, status_md, audio_out]

    send_btn.click(user_submit,  submit_inputs, submit_outputs)
    msg_box.submit(user_submit,  submit_inputs, submit_outputs)
    clear_btn.click(lambda: ([], "", "*Ready*", None), None, submit_outputs)
    transcribe_btn.click(handle_audio_input, [audio_in, msg_box], msg_box)

    for btn, ex in example_btns:
        btn.click(lambda e=ex: e, None, msg_box)

demo.queue().launch(share=True)